# `POGGSEMENTUtil` usage examples

The `POGGSEMENTUtil` class contains a number of functions that are useful for manipulating, comparing, and printing SEMENT structures.

[See full module documentation here](project:/apidocs/pogg/pogg.semantic_composition.sement_util.POGGSEMENTUtil.md)

The most useful function in this module is `build_isomorphism_report` because it checks whether two SEMENTs are isomorphic (i.e. have the same structure and "mean the same thing") and then prints a report on any discrepancies found between the SEMENTs.

Most other functions in this module are either helper functions for `build_isomorphism_report` or are used within functions in the semantic composition classes.

If you just want an example of what an isomorphism report would look like, skip to the [`build_isomorphism_report` example](#build-isomorphism-report-example). If you have an interest in how the helper functions each contribute to this result, then the rest of the page builds up to that point.

For the following examples, we need these imports:

In [4]:
import pprint
import pogg.my_delphin.sementcodecs as sementcodecs
from pogg.semantic_composition.sement_util import POGGSEMENTUtil



## `group_equalities` example

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.POGGSEMENTUtil.group_equalities
:summary:
```

:::{info} Function details
:collapsible:

````{py:function} group_equalities(equalities)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.POGGSEMENTUtil.group_equalities
```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.POGGSEMENTUtil.group_equalities
```
````
:::

In [5]:
eqs = [("x1", "x2"), ("x3", "x4"), ("x1", "x4"), ("x5", "x6")]
grouped_eqs = POGGSEMENTUtil.group_equalities(eqs)
print(grouped_eqs)

[{'x5', 'x6'}, {'x4', 'x1', 'x2', 'x3'}]


## `get_most_specified_variable` example

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.POGGSEMENTUtil.group_equalities
:summary:
```

:::{info} Function details
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.group_equalities
:collapsible:

````{py:function} get_most_specified_variable(eq_vars)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.get_most_specified_variable

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.get_most_specified_variable
```
````
:::


In [6]:
eq_set = {'u1', 'x2', 'i3'}

# should return 'x2'
POGGSEMENTUtil.get_most_specified_variable(eq_set)

'x2'

## `overwrite_eqs` example

Example of `overwrite_eqs` being called on a SEMENT for *tasty cookie*. In the initial SEMENT, there is an EQ between the `ARG0` of *cookie* (`x1`) and the `ARG1` of *tasty*, because the `ARG1` of *cookie* (i.e. the thing that is tasty) is plugged by the intrinsic variable of *cookie*.

The `overwrite_eqs` function chooses one of these variables as the representative for the EQ (here, `x1`) and overwrites all instances of `x4` to also be ``x1``. This enables conversion to an MRS object in order to send the structure to the ERG for text generation.

:::{info} Function details
:collapsible:
````{py:function} overwrite_eqs(sement)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.overwrite_eqs

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.overwrite_eqs
```
````
:::

In [7]:
# SEMENT for "tasty cookie" before EQs are overwritten
sement_string = """[
    TOP: h0
    INDEX: x1
    RELS: <
        [ _tasty_a_1 LBL: h2 ARG0: e3 ARG1: x4 ]
        [ _cookie_n_1 LBL: h0 ARG0: x1 ] >
    EQS: < x1 eq x4 >
]"""

# convert string into SEMENT object
original_sement = sementcodecs.decode(sement_string)

new_sement = POGGSEMENTUtil.overwrite_eqs(original_sement)

# encode the new_sement object into a string and print it
print(sementcodecs.encode(new_sement, indent=True))

NameError: name 'group_equalities' is not defined

## `check_if_quantified` example

This function is used in cases where it's necessary to ensure that a SEMENT is quantified before proceeding with composition. For example, the argument of a verb must be plugged with a quantified noun (plus possible adjuncts). That is, it cannot be plugged with a SEMENT whose `RELS` list only contains one EP for *cookie*, but a SEMENT whose `RELS` list contains an EP for *cookie* and some quantifier.

:::{info} Function details
:collapsible:
````{py:function} check_if_quantified(check_sement)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.check_if_quantified

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.check_if_quantified
```
````
:::

In [5]:
unquant_cookie = """[ TOP: h0
    INDEX: x1
    RELS: < [ _cookie_n_1 LBL: h0 ARG0: x1 ] >
]"""
unquant_cookie_sement_obj = sementcodecs.decode(unquant_cookie)

POGGSEMENTUtil.check_if_quantified(unquant_cookie_sement_obj)

False

In [6]:
quant_cookie = """[ TOP: h6
    INDEX: x1
    RELS: <
        [ _udef_q LBL: h0 ARG0: x1 RSTR: h2 BODY: h3 ]
        [ _cookie_n_1 LBL: h4 ARG0: x5 ] >
    EQS: < x1 eq x5 >
    SLOTS: < BODY: h3 >
    HCONS: < h2 qeq h4 > ]"""

quant_cookie_sement_obj = sementcodecs.decode(quant_cookie)

POGGSEMENTUtil.check_if_quantified(quant_cookie_sement_obj)

True

## `is_sement_isomorphic` example

 Here are two examples of `is_sement_isomorphic`. In the first one, the SEMENTs are isomorphic, but just have different variable names and different orders on the `RELS` lists, so the check returns `True`.

:::{info} Function details
:collapsible:
````{py:function} is_sement_isomorphic(s1: pogg.my_delphin.my_delphin.SEMENT, s2: pogg.my_delphin.my_delphin.SEMENT) -> bool
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.is_sement_isomorphic

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.is_sement_isomorphic
```
````
:::

In [7]:
sement1 = """[
    TOP: h0101
    INDEX: e1101 [e NUM: sg]
    RELS: <
        [ _a_q LBL: h10 ARG0: x11 RSTR: h12 BODY: h13]
        [ _cookie_n_1 LBL: h6 ARG0: x7 ]
        [ _give_v_1 LBL: h0 ARG0: e1 ARG1: i2 ARG2: u3 ARG3: i4 ]
    >
    EQS: < x11 eq x7 eq i2 h0101 eq h0 e1 eq e1101 >
    SLOTS: < ARG2: u3 ARG3: i4 >
    HCONS: < h12 qeq h6 >

]"""

sement2 = """[
    TOP: h011
    INDEX: e111 [e NUM: sg]
    RELS: <
        [ _give_v_1 LBL: h011 ARG0: e111 ARG1: i211 ARG2: u311 ARG3: i411 ]
        [ _a_q LBL: h1011 ARG0: x1111 RSTR: h1211 BODY: h1311]
        [ _cookie_n_1 LBL: h611 ARG0: x711 ]
    >
    EQS: < x1111 eq x711 x711 eq i211 >
    SLOTS: < ARG2: u311 ARG3: i411 >
    HCONS: < h1211 qeq h611 >

]"""

sement1_obj = sementcodecs.decode(sement1)
sement2_obj = sementcodecs.decode(sement2)

POGGSEMENTUtil.is_sement_isomorphic(sement1_obj, sement2_obj)

True

In the second example, the SEMENTs are almost isomorphic, but the `TOP` is identified with `_cookie_n_1.LBL`, rather than `_give_v_1.LBL`, causing a discrepancy.

In [8]:
sement3 = """[
    TOP: h611
    INDEX: e111 [e NUM: sg]
    RELS: <
        [ _give_v_1 LBL: h011 ARG0: e111 ARG1: i211 ARG2: u311 ARG3: i411 ]
        [ _a_q LBL: h1011 ARG0: x1111 RSTR: h1211 BODY: h1311]
        [ _cookie_n_1 LBL: h611 ARG0: x711 ]
    >
    EQS: < x1111 eq x711 x711 eq i2 >
    SLOTS: < ARG2: u311 ARG3: i411 >
    HCONS: < h1211 qeq h611 >

]"""

sement3_obj = sementcodecs.decode(sement3)

# compare again to the first SEMENT
POGGSEMENTUtil.is_sement_isomorphic(sement1_obj, sement3_obj)

False

Detecting these differences at a glance is difficult, and the function `build_isomorphism_report` aims to alleviate this, but before going over an example of that a number of helper functions are used within `build_isomorphism_report` so let's go over those first.

## `create_variable_roles_dict` example

This function creates a dictionary where each key is a variable in the SEMENT and the value is the set of semantic roles that variable fills. Having this information will help when comparing two SEMENTs.

:::{info} Function details
:collapsible:
````{py:function} create_variable_roles_dict(sement)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.create_variable_roles_dict

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.create_variable_roles_dict
```
````
:::

Below is an example of calling it on a SEMENT for *a tasty cookie*

In [9]:
sement_string = """[ TOP: h9
                  INDEX: x4
                  RELS: < [ _tasty_a_1 LBL: h0 ARG0: e1 ARG1: x4 ]
                          [ _cookie_n_1 LBL: h0 ARG0: x4 ]
                          [ _a_q LBL: h5 ARG0: x4 RSTR: h7 BODY: h8 ] >
                  SLOTS: < BODY: h8 >
                  HCONS: < h7 qeq h0 > ]"""

# convert to object from string
sement_object = sementcodecs.decode(sement_string)

POGGSEMENTUtil.create_variable_roles_dict(sement_object)

{'h9': ['TOP'],
 'x4': ['INDEX', '_a_q.ARG0', '_cookie_n_1.ARG0', '_tasty_a_1.ARG1'],
 'h0': ['_cookie_n_1.LBL', '_tasty_a_1.LBL'],
 'e1': ['_tasty_a_1.ARG0'],
 'h5': ['_a_q.LBL'],
 'h7': ['_a_q.RSTR'],
 'h8': ['_a_q.BODY']}

## `create_hcons_list` example

This function creates a list of "hcons entries" for a given SEMENT. An "hcon entry" details which semantic roles are paricipating in a handle constraint. Having this information enables comparison of handle constraints across SEMENTs.

:::{info} Function details
:collapsible:
````{py:function} create_hcons_list(sement)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.create_hcons_list

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.create_hcons_list
```
````
:::

Below is an example of calling it on a SEMENT that roughly corresponds to the verb phrase *know the cat sleeps*

In [10]:
sement_string = """[ TOP: h0
  INDEX: e1
  RELS: < [ _know_v_1 LBL: h0 ARG0: e1 ARG1: i2 ARG2: u3 ]
          [ _the_q LBL: h5 ARG0: x10 RSTR: h7 BODY: h8 ]
          [ _cat_n_1 LBL: h9 ARG0: x10 ]
          [ _sleep_v_1 LBL: h11 ARG0: e12 ARG1: x10 ] >
  SLOTS: < ARG1: i2 >
  HCONS: < h7 qeq h9 u3 qeq h11 > ]"""

# convert to object from string
sement_object = sementcodecs.decode(sement_string)

POGGSEMENTUtil.create_hcons_list(sement_object)

[{'hi_role_set': ['_the_q.RSTR'],
  'lo_role_set': ['_cat_n_1.LBL'],
  'hi_var': 'h7',
  'lo_var': 'h9'},
 {'hi_role_set': ['_know_v_1.ARG2'],
  'lo_role_set': ['_sleep_v_1.LBL'],
  'hi_var': 'u3',
  'lo_var': 'h11'}]

## `find_slot_overlaps` example

This function compares slots lists across two SEMENTs to detect differences when isomorphism checks fail.

:::{info} Function details
:collapsible:
````{py:function} find_slot_overlaps(gold_sement, actual_sement)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.find_slot_overlaps

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.find_slot_overlaps
```
````
:::

Below is an example of calling `find_slot_overlaps` on two SEMENTs for *believe the cat sleeps*. In the broken one, there is a handle constraint between `_believe_v_1.ARG3` and `_sleep_v_1.LBL`, which is the incorrect slot for this constraint, causing the slots list to be incorrect. [^1]

[^1]: Technically, they are both broken because we want the two argument version of `_believe_v_1`, not the three argument version, but using the three argument version is better for the examples.


In [11]:
gold_sement_str = """[ TOP: h0
  INDEX: e1
  RELS: < [ _believe_v_1 LBL: h0 ARG0: e1 ARG1: i2 ARG2: u3 ARG3: h4 ]
          [ _the_q LBL: h5 ARG0: x10 RSTR: h7 BODY: h8 ]
          [ _cat_n_1 LBL: h9 ARG0: x10 ]
          [ _sleep_v_1 LBL: h11 ARG0: e12 ARG1: x10 ] >
  SLOTS: < ARG1: i2 ARG3: h4 >
  HCONS: < h7 qeq h9 u3 qeq h11 > ]"""

broken_sement_str = """[ TOP: h00
  INDEX: e01
  RELS: < [ _believe_v_1 LBL: h00 ARG0: e01 ARG1: i02 ARG2: u03 ARG3: h04 ]
          [ _the_q LBL: h05 ARG0: x010 RSTR: h07 BODY: h08 ]
          [ _cat_n_1 LBL: h09 ARG0: x010 ]
          [ _sleep_v_1 LBL: h011 ARG0: e012 ARG1: x010 ] >
  SLOTS: < ARG1: i02 ARG2: u03 >
  HCONS: < h07 qeq h09 h04 qeq h011 > ]"""

gold_sement_obj = sementcodecs.decode(gold_sement_str)
broken_sement_obj = sementcodecs.decode(broken_sement_str)

# the function returns three lists, store each of them
overlapping_slots, gold_slots, broken_slots = POGGSEMENTUtil.find_slot_overlaps(gold_sement_obj, broken_sement_obj)


# print resulting slot lists
print("OVERLAPPING: {}".format(overlapping_slots))
print("GOLD: {}".format(gold_slots))
print("BROKEN: {}".format(broken_slots))

OVERLAPPING: [{'slot': ['_believe_v_1.ARG1'], 'gold_var': 'i2', 'actual_var': 'i02'}]
GOLD: [{'slot': ['_believe_v_1.ARG3'], 'gold_var': 'h4'}]
BROKEN: [{'slot': ['_believe_v_1.ARG2'], 'actual_var': 'u03'}]


## `find_var_eq_overlaps` example

This function compares semantic role identities across two SEMENTs to detect differences when isomorphism checks fail. What is meant by "semantic role identities" is that, for example, in the phrase *a tasty cookie*, the `_tasty_a_.ARG1` (i.e. the thing that is tasty) and `_cookie_n_1.ARG0` (i.e. the cookie instance) are linked together. For two SEMENTs to be isomorphic all identities between semantic roles should be the same and this function builds lists of which sets of semantic role identities overlap and which ones don't.

:::{info} Function details
:collapsible:
````{py:function} find_var_eq_overlaps(gold_sement, actual_sement)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.find_var_eq_overlaps

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.find_var_eq_overlaps
```
````
:::

This function produces three lists:
- `overlap_eqs` -- semantic role identities present in both SEMENTs
- `gold_only_eqs` -- semantic role identities present in only the "gold" SEMENT (e.g. the SEMENT you're trying to emulate)
- `actual_only_eqs` -- semantic role identities present in only the "actual" (e.g. one produced by the system) SEMENT

Here is an example of a "role identity set":

```
{
    "eq_set": {"_a_q.ARG0", "_cat_n_1.ARG0", "_cozy_a_1.ARG1"}
    "gold_var": 'x1',
    "actual_var": 'x2'
}
```

What this shows is that in both SEMENTs `_a_q.ARG0`, `_cat_n_1.ARG0`, and `_cozy_a_1.ARG1` EP are identified with each other, but that in the gold SEMENT the variable filling all these slots is called `x1` but in the actual (where "actual" roughly means the one being composed with the goal to match the gold) the variable is called `x2`. So, despite the different variable names, the semantic role equivalency matches.

If two SEMENTs are isomorphic, the `gold_only_eqs` and `actual_only_eqs` lists will be empty, but when the SEMENTs are not isomorphic, the sets of semantic role identities may not match so these lists will help pinpoint where the differences lie.

### Worked Example

Below is a worked example of calling `find_var_eq_overlaps` on two SEMENTs for *tasty cookie*. In the broken one, there should be an equivalence between the `LBL` of each EP, but it's missing.

First, let's gather the lists of role equivalence sets

In [12]:
gold_tasty_cookie = """[ TOP: h0
                  INDEX: x4
                  RELS: < [ _tasty_a_1 LBL: h0 ARG0: e1 ARG1: x4 ]
                          [ _cookie_n_1 LBL: h0 ARG0: x4 ] > ]"""

broken_tasty_cookie = """[ TOP: h03
  INDEX: x04
  RELS: < [ _tasty_a_1 LBL: h00 ARG0: e01 ARG1: x04 ]
          [ _cookie_n_1 LBL: h03 ARG0: x04 ] > ]"""

gold_sement_obj = sementcodecs.decode(gold_tasty_cookie)
broken_sement_obj = sementcodecs.decode(broken_tasty_cookie)

# the function returns three lists, store each of them
overlapping_eqs, gold_eqs, broken_eqs = POGGSEMENTUtil.find_var_eq_overlaps(gold_sement_obj, broken_sement_obj)

Now, let's look at the overlapping sets. Notice that the first set has multiple semantic roles in it (the `INDEX`, `_cookie_n_.ARG0`, and the `_tasty_a_1.ARG1`). But the second set just contains one role, `_tasty_a_1.ARG0`. Even though it feels odd to call that a set of "identities" it does indicate to us that in both SEMENTs this role is not identified with any other role.

In [13]:
# shows the eqs present in both SEMENTS
pprint.pp(overlapping_eqs)

[{'eq_set': ['INDEX', '_cookie_n_1.ARG0', '_tasty_a_1.ARG1'],
  'gold_var': 'x4',
  'actual_var': 'x04'},
 {'eq_set': ['_tasty_a_1.ARG0'], 'gold_var': 'e1', 'actual_var': 'e01'}]


Next, let's look at the role identities only present in the gold SEMENT. Here, we see the identity between the `LBL` of each EP as mentioned above, as well as the fact that the `TOP` is a member of this set.

In [14]:
# shows the eqs present in only in the gold SEMENT
pprint.pp(gold_eqs)

[{'eq_set': ['TOP', '_cookie_n_1.LBL', '_tasty_a_1.LBL'], 'gold_var': 'h0'}]


Last, let's look at the role identities unique to the broken SEMENT. Here, we have two sets. The first one shows that the `TOP` is identified with `_cookie_n_1.LBL`. The second one shows that `_tasty_a_1.LBL` is not identified with anything, which is in contrast to what we saw in the gold one.

In [15]:
# shows the eqs present in only in the "actual" SEMENT
pprint.pp(broken_eqs)

[{'eq_set': ['TOP', '_cookie_n_1.LBL'], 'actual_var': 'h03'},
 {'eq_set': ['_tasty_a_1.LBL'], 'actual_var': 'h00'}]


## `find_hcons_overlaps` example

This function compares handle constraints across two SEMENTs to detect differences when isomorphism checks fail.

:::{info} Function details
:collapsible:
````{py:function} find_hcons_overlaps(gold_sement, actual_sement)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.find_hcons_overlaps

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.find_hcons_overlaps
```
````
:::

This function produces three lists:
- `overlap_hcons` -- handle constraints present in both SEMENTs
- `gold_only_hcons` -- handle constraints present in only the "gold" SEMENT (e.g. the SEMENT you're trying to emulate)
- `actual_only_hcons` -- handle constraints present in only the "actual" (e.g. one produced by the system) SEMENT

Here is an example of an entry in the `overlap_hcons` list:

```
 {
    "hi_role_set": {"_a_q.RSTR"},
    "lo_role_set": {"_cookie_n_1.LBL", "_tasty_a_1.LBL"},
    "hi_gold_var": "h0",
    "lo_gold_var": "h1",
    "hi_actual_var": "h00",
    "lo_actual_var": "h01",
}
```

What this shows is that in both SEMENTs `_a_q.RSTR` is the high handle in a QEQ relationship to both `_cookie_n_1.LBL` and `_tasty_a_1.LBL`, which serve as the low handles in conjunction. But the values of the handles in each SEMENT are different, even though the shape of the handle constraint is the same in both.

If two SEMENts are isomorphic, the `gold_only_hcons` and `actual_only_hcons` lists will be empty, but when the SEMENTs are not isomorphic, the sets of handle constraints may not match so these lists will help pinpoint where the differences lie.

### Worked example

Below is a worked example of calling `find_var_eq_overlaps` on two SEMENTs for *believe the cat sleeps*. In the broken one, there should be a QEQ between `_believe_v_1.ARG2` EP and the `LBL` of the `_sleep_v_1.LBL`, but it's actually set between the wrong arguments. [^1]

First, let's gather the lists of handle constraints.



In [16]:
believe_the_cat_sleeps = """[ TOP: h0
  INDEX: e1
  RELS: < [ _believe_v_1 LBL: h0 ARG0: e1 ARG1: i2 ARG2: u3 ARG3: h4 ]
          [ the_q LBL: h5 ARG0: x10 RSTR: h7 BODY: h8 ]
          [ _cat_n_1 LBL: h9 ARG0: x10 ]
          [ _sleep_v_1 LBL: h11 ARG0: e12 ARG1: x10 ] >
  SLOTS: < ARG1: i2 ARG3: h4 >
  HCONS: < h7 qeq h9 u3 qeq h11 > ]"""

broken_believe_the_cat_sleeps = """[ TOP: h00
  INDEX: e01
  RELS: < [ _believe_v_1 LBL: h00 ARG0: e01 ARG1: i02 ARG2: u03 ARG3: h04 ]
          [ the_q LBL: h05 ARG0: x010 RSTR: h07 BODY: h08 ]
          [ _cat_n_1 LBL: h09 ARG0: x010 ]
          [ _sleep_v_1 LBL: h011 ARG0: e012 ARG1: x010 ] >
  SLOTS: < ARG1: i02 >
  HCONS: < h07 qeq h09 h04 qeq h011 > ]"""

gold_sement_obj = sementcodecs.decode(believe_the_cat_sleeps)
broken_sement_obj = sementcodecs.decode(broken_believe_the_cat_sleeps)

overlapping_hcons, gold_hcons, broken_hcons = (POGGSEMENTUtil.find_hcons_overlaps(gold_sement_obj, broken_sement_obj))

Now, let's look at the overlapping handle constraints.

In [17]:
# shows the hcons present in both SEMENTS
pprint.pp(overlapping_hcons)

[{'hi_role_set': ['the_q.RSTR'],
  'lo_role_set': ['_cat_n_1.LBL'],
  'gold_hi_var': 'h7',
  'gold_lo_var': 'h9',
  'actual_hi_var': 'h07',
  'actual_lo_var': 'h09'}]


Here we see that in both SEMENTs we have the QEQ relationship between `_the_q.RSTR` and `_cat_n_1.LBL`.

Next, let's look at the handle constraints only present in the gold SEMENT.

In [18]:
# shows the hcons present in only in the gold SEMENT
pprint.pp(gold_hcons)

[{'hi_role_set': ['_believe_v_1.ARG2'],
  'lo_role_set': ['_sleep_v_1.LBL'],
  'gold_hi_var': 'u3',
  'gold_lo_var': 'h11'}]


Here, we see the desired QEQ relationship between `_believe_v_1.ARG2` and `_sleep_v_1.LBL`.

Last, let's look at the handle constraints unique to the broken SEMENT.



In [19]:
# shows the hcons present in only in the "actual" SEMENT
pprint.pp(broken_hcons)

[{'hi_role_set': ['_believe_v_1.ARG3'],
  'lo_role_set': ['_sleep_v_1.LBL'],
  'actual_hi_var': 'h04',
  'actual_lo_var': 'h011'}]


This pinpoints the error in the broken version: the high handle in the QEQ is the `ARG3` of the `_believe_v_1` EP rather than the `ARG2`.

## `_build_overlap_slots_table` example

This is a helper function used inside of `build_isomorphism_report` and shouldn't be used directly, but here is an example of what it outputs.

:::{info} Function details
:collapsible:
````{py:function} _build_overlap_slots_table(overlap_slots)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_overlap_slots_table

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_overlap_slots_table
```
````
:::

In [20]:
overlapping_slots = [{'slot': ['_believe_v_1.ARG1'],
        'gold_var': 'i2',
        'actual_var': 'i02'},
    {'slot': ['_believe_v_1.ARG3'],
        'gold_var': 'h4',
        'actual_var': 'h04'}]

print(POGGSEMENTUtil._build_overlap_slots_table(overlapping_slots))

Slot Name              Gold Var    Actual Var
---------------------  ----------  ------------
['_believe_v_1.ARG1']  i2          i02
['_believe_v_1.ARG3']  h4          h04


## `_build_nonoverlap_slots_table` example

This is a helper function used inside of `build_isomorphism_report` and shouldn't be used directly, but here is an example of what it outputs.

:::{info} Function details
:collapsible:
````{py:function} _build_nonoverlap_slots_table(nonoverlap_slots, table_type)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_nonoverlap_slots_table

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_nonoverlap_slots_table
```
````
:::


In [21]:
nonoverlapping_slots = [{'slot': ['_cozy_a_1.ARG1'], 'gold_var': 'i2'}]

print(POGGSEMENTUtil._build_nonoverlap_slots_table(nonoverlapping_slots, "gold"))

Slot Name           Gold Var
------------------  ----------
['_cozy_a_1.ARG1']  i2


## `_build_overlap_eqs_table` example

This is a helper function used inside of `build_isomorphism_report` and shouldn't be used directly, but here is an example of what it outputs.

:::{info} Function details
:collapsible:
````{py:function} _build_overlap_eqs_table(overlap_eqs)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_overlap_eqs_table

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_overlap_eqs_table
```
````
:::

In [22]:
overlapping_role_eqs = [{'eq_set': ['INDEX', '_cookie_n_1.ARG0','_tasty_a_1.ARG1'],
        'gold_var': 'x4',
        'actual_var': 'x04'},
    {'eq_set': ['_tasty_a_1.ARG0'],
        'gold_var': 'e1',
        'actual_var': 'e01'}]

print(POGGSEMENTUtil._build_overlap_eqs_table(overlapping_role_eqs))

Role Set                                          Gold Var    Actual Var
------------------------------------------------  ----------  ------------
['INDEX', '_cookie_n_1.ARG0', '_tasty_a_1.ARG1']  x4          x04
['_tasty_a_1.ARG0']                               e1          e01


## `_build_nonoverlap_eqs_table` example

This is a helper function used inside of `build_isomorphism_report` and shouldn't be used directly, but here is an example of what it outputs.

:::{info} Function details
:collapsible:
````{py:function} _build_nonoverlap_eqs_table(nonoverlap_eqs, table_type)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_nonoverlap_eqs_table

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_nonoverlap_eqs_table
```
````
:::

In [23]:
nonoverlapping_role_eqs = [{'eq_set': ['TOP', '_cookie_n_1.LBL', '_tasty_a_1.LBL'], 'gold_var': 'h0'}]

print(POGGSEMENTUtil._build_nonoverlap_eqs_table(nonoverlapping_role_eqs, "gold"))

Role Set                                      Gold Var
--------------------------------------------  ----------
['TOP', '_cookie_n_1.LBL', '_tasty_a_1.LBL']  h0


## `_build_overlap_hcons_table` example

This is a helper function used inside of `build_isomorphism_report` and shouldn't be used directly, but here is an example of what it outputs.

:::{info} Function details
:collapsible:
````{py:function} _build_overlap_hcons_table(overlap_hcons)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_overlap_hcons_table

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_overlap_hcons_table
```
````
:::

In [24]:
overlapping_hcons = [{'hi_role_set': ['the_q.RSTR'],
        'lo_role_set': ['_cat_n_1.LBL', '_cute_a_1.LBL'],
        'gold_hi_var': 'h7',
        'gold_lo_var': 'h9',
        'actual_hi_var': 'h07',
        'actual_lo_var': 'h09'}]

print(POGGSEMENTUtil._build_overlap_hcons_table(overlapping_hcons))

Hi Role Set     Lo Role Set                        Gold QEQ    Actual QEQ
--------------  ---------------------------------  ----------  ------------
['the_q.RSTR']  ['_cat_n_1.LBL', '_cute_a_1.LBL']  h7 qeq h9   h07 qeq h09


## `_build_nonoverlap_hcons_table` example

This is a helper function used inside of `build_isomorphism_report` and shouldn't be used directly, but here is an example of what it outputs.

:::{info} Function details
:collapsible:
````{py:function} _build_nonoverlap_hcons_table(nonoverlap_hcons, table_type)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_nonoverlap_hcons_table

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil._build_nonoverlap_hcons_table
```
````
:::

In [25]:
nonoverlapping_role_hcons = [{'hi_role_set': ['_believe_v_1.ARG2'],
        'lo_role_set': ['_sleep_v_1.LBL'],
        'gold_hi_var': 'u3',
        'gold_lo_var': 'h11'}]

print(POGGSEMENTUtil._build_nonoverlap_hcons_table(nonoverlapping_role_hcons, "gold"))

Hi Role Set            Lo Role Set         Gold QEQ
---------------------  ------------------  ----------
['_believe_v_1.ARG2']  ['_sleep_v_1.LBL']  u3 qeq h11


## `build_isomorphism_report` example

This function builds a report that details the consistences and discrepancies between two sements.

:::{info} Function details
:collapsible:
````{py:function} build_isomorphism_report(gold_sement, actual_sement)
:canonical: pogg.semantic_composition.sement_util.POGGSEMENTUtil.build_isomorphism_report

```{autodoc2-docstring} pogg.semantic_composition.sement_util.POGGSEMENTUtil.build_isomorphism_report
```
````
:::

Here is an example of an isomorphism report between a SEMENT for *believe the cat sleeps* and a broken version of the SEMENT. In the broken one, there is a handle constraint between `_believe_v_1.ARG3` and `_sleep_v_1.LBL`, but the first member of the constraint should instead be `_believe_v_1.ARG2`. Secondly, `_the_q.ARG0`, `_cat_n_1.ARG0`, and `_sleep_v_1.ARG1` should all be filled by the same variable, but in the broken one `sleep_v_1.ARG1` isn't included in the identity.

In [26]:
gold_sement_string = """[ TOP: h0
  INDEX: e1
  RELS: < [ _believe_v_1 LBL: h0 ARG0: e1 ARG1: i2 ARG2: u3 ARG3: h4 ]
          [ _the_q LBL: h5 ARG0: x10 RSTR: h7 BODY: h8 ]
          [ _cat_n_1 LBL: h9 ARG0: x10 ]
          [ _sleep_v_1 LBL: h11 ARG0: e12 ARG1: x10 ] >
  SLOTS: < ARG1: i2 ARG3: h4 >
  HCONS: < h7 qeq h9 u3 qeq h11 > ]"""

broken_sement_string = """[ TOP: h00
  INDEX: e2
  RELS: < [ _believe_v_1 LBL: h00 ARG0: e2 ARG1: i4 ARG2: u6 ARG3: h8 ]
          [ _the_q LBL: h10 ARG0: x20 RSTR: h14 BODY: h16 ]
          [ _cat_n_1 LBL: h18 ARG0: x20 ]
          [ _sleep_v_1 LBL: h22 ARG0: e24 ARG1: x1010 ] >
  SLOTS: < ARG1: i4 ARG2: u6 >
  HCONS: < h14 qeq h18 h8 qeq h22 > ]"""

gold_sement = sementcodecs.decode(gold_sement_string)
broken_sement = sementcodecs.decode(broken_sement_string)

report = POGGSEMENTUtil.build_isomorphism_report(gold_sement, broken_sement)
print(report)

--- GOLD SEMENT ---
[ TOP: h0
  INDEX: e1
  RELS: < [ _believe_v_1 LBL: h0 ARG0: e1 ARG1: i2 ARG2: u3 ARG3: h4 ]
          [ _the_q LBL: h5 ARG0: x10 RSTR: h7 BODY: h8 ]
          [ _cat_n_1 LBL: h9 ARG0: x10 ]
          [ _sleep_v_1 LBL: h11 ARG0: e12 ARG1: x10 ] >
  HCONS: < h7 qeq h9 u3 qeq h11 >
  SLOTS: < ARG1: i2 ARG3: h4 > ]

--- ACTUAL SEMENT ---
[ TOP: h00
  INDEX: e2
  RELS: < [ _believe_v_1 LBL: h00 ARG0: e2 ARG1: i4 ARG2: u6 ARG3: h8 ]
          [ _the_q LBL: h10 ARG0: x20 RSTR: h14 BODY: h16 ]
          [ _cat_n_1 LBL: h18 ARG0: x20 ]
          [ _sleep_v_1 LBL: h22 ARG0: e24 ARG1: x1010 ] >
  HCONS: < h14 qeq h18 h8 qeq h22 >
  SLOTS: < ARG1: i4 ARG2: u6 > ]


=== DISCREPANCIES ===

SLOT DISCREPANCIES
^^^^^^^^^^^^^^^^^^
GOLD ONLY
Slot Name              Gold Var
---------------------  ----------
['_believe_v_1.ARG3']  h4

ACTUAL ONLY
Slot Name              Actual Var
---------------------  ------------
['_believe_v_1.ARG2']  u6


SEMANTIC ROLE IDENTITY DISCREPANCIES
^^^^^^